# Customer Churn Prediction - Stage 7: Model Improvement
**Author:** B.Tech ECE Student ML Project  
**Objective:** Handle class imbalance using `class_weight='balanced'` and tune hyperparameters using 5-fold Stratified `GridSearchCV`. Compare Before vs. After metrics to demonstrate business value.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Load datasets
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/y_test.csv').values.ravel()
print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

### 1. Hyperparameter Tuning with 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid for Random Forest
rf_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [5, 8, 12],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print(f"Best RF Parameters: {rf_grid.best_params_}")
print(f"Best RF CV ROC-AUC: {rf_grid.best_score_:.4f}")

### 2. Evaluating Tuned Random Forest on Test Data

In [ ]:
best_rf = rf_grid.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
y_proba_tuned = best_rf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()

print(f"Tuned Random Forest Test Accuracy:  {accuracy_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Tuned Random Forest Test Precision: {precision_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Tuned Random Forest Test Recall:    {recall_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Tuned Random Forest Test F1-Score:  {f1_score(y_test, y_pred_tuned)*100:.2f}%")
print(f"Tuned Random Forest Test ROC-AUC:   {roc_auc_score(y_test, y_proba_tuned):.4f}")
print(f"\nCaught Churners (TP): {tp} / {tp+fn} ({tp/(tp+fn)*100:.1f}% of all actual churners caught!)")
print(f"Missed Churners (FN): {fn}")